# IDP Curriculum Generation Pipeline

Reusable Kaggle notebook for generating offline educational curriculum artifacts from **raw textbook PDFs**.

Corrected pipeline:

```text
Raw PDF textbooks
  -> Docling PDF extraction inside Kaggle
  -> structured sections, tables, captions, equations
  -> Gemma 4 12B IT GGUF via llama.cpp
  -> JSON validation, retry, cache, checkpoint
  -> generated curriculum pack
```

Important: this notebook does **not** require pre-generated `structured_chapter.json` inputs. It creates structured content from raw PDFs using Docling.

Important: there is **no separate vision model**. Gemma processes Docling-extracted text, captions, figure references, tables, and equations. If image-level visual reasoning is needed later, it can be added as a separate sprint, but this notebook intentionally keeps one content model.

## Architecture Diagram

```text
/kaggle/input/<raw-pdf-dataset>/
        |
        v
PDF Discovery
        |
        v
Docling PDF Converter
        |
        v
StructuredChapter
  - chapter title
  - sections
  - tables
  - equations
  - figure references
  - captions
        |
        v
Section Builder
  - target: 500-1500 words
  - never sends whole textbook/chapter to Gemma
        |
        v
Gemma 4 12B IT GGUF (llama.cpp)
        |
        v
Artifact Generators
  - concepts
  - summaries
  - glossary
  - quizzes
  - flashcards
  - learning objectives
  - misconceptions
  - teacher notes
  - investigations
        |
        v
Validation + Retry + JSON Repair + Cache
        |
        v
/kaggle/working/idp_curriculum_generation/generated_pack.zip
```

## 00_PROJECT_CONFIG

All configurable values live here. Do not hardcode paths or model names elsewhere.

In [ ]:
CONFIG = {
    # Kaggle dataset containing raw PDF textbooks.
    # If unsure, keep /kaggle/input. Discovery will recursively find TEXTBOOKS/**.pdf.
    # Example: /kaggle/input/<dataset-name>/TEXTBOOKS/mathematics/class 8/Chapter.pdf
    "pdf_dataset_root": "/kaggle/input/textbooks",
    "output_root": "/kaggle/working/idp_curriculum_generation",

    # Process subject-by-subject. Use [] for all subjects.
    # Examples: ["mathematics"], ["science"], ["social_science"], ["biology", "physics", "chemistry"]
    "include_subjects": [],

    # Process grade-by-grade. Use [] for all grades.
    # Examples: ["8"], ["6", "7"], ["10"]
    "include_grades": [],

    # Optional filename filter for smoke tests.
    # Examples: "photosynthesis", "proportional", "force"
    "include_pdf_name_contains": "",

    # Processing limits for testing and batch runs.
    # Example: set max_pdfs=5 for a quick run.
    "max_pdfs": None,
    "max_sections": None,
    "debug_discovery": True,

    # Kaggle accelerator recommendation. Select T4 in the Kaggle UI.
    "recommended_accelerator": "T4",

    # Content model requested for Kaggle generation.
    "content_model_repo": "lmstudio-community/gemma-4-12B-it-GGUF",
    "content_model_file": "gemma-4-12B-it-Q4_K_M.gguf",

    # Optional fallback, only used if primary model load fails.
    "fallback_content_model_repo": "lmstudio-community/gemma-4-E2B-it-GGUF",
    "fallback_content_model_file": "gemma-4-E2B-it-Q4_K_M.gguf",

    # Generation controls.
    "generate_concepts": True,
    "generate_learning_objectives": True,
    "generate_glossary": True,
    "generate_summary": True,
    "generate_detailed_explanation": True,
    "generate_misconceptions": True,
    "generate_mcq_quiz": True,
    "generate_short_answer": True,
    "generate_flashcards": True,
    "generate_concept_relationships": True,
    "generate_investigations": True,
    "generate_teacher_notes": True,
    "generate_prerequisites": True,
    "generate_difficulty_analysis": True,

    # Docling controls.
    "docling_do_ocr": False,
    "docling_do_table_structure": True,
    "docling_generate_picture_images": False,
    "docling_generate_page_images": False,

    # Section sizing. Sections are sent to Gemma independently.
    "min_section_words": 500,
    "max_section_words": 1500,

    # llama.cpp generation settings.
    "temperature": 0.2,
    "top_p": 0.9,
    "max_tokens": 900,
    "context_size": 8192,
    "n_gpu_layers": -1,
    "n_threads": 4,
    "max_retries": 3,

    # Reliability.
    "enable_validation": True,
    "save_intermediate_outputs": True,
    "resume": True,
    "artifact_zip_name": "generated_pack.zip"
}


## 01_ENVIRONMENT_SETUP

Install and verify dependencies. Kaggle internet must be enabled for first-time package/model downloads.

In [ ]:
!pip -q install docling llama-cpp-python jsonschema pandas pillow opencv-python tqdm sentencepiece accelerate transformers

import importlib, json, logging, os, re, time, hashlib, zipfile
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
from tqdm.auto import tqdm
from jsonschema import validate as jsonschema_validate

for module_name in ["docling", "llama_cpp", "jsonschema", "pandas", "PIL", "cv2"]:
    importlib.import_module(module_name)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("idp-curriculum-generation")

OUTPUT_ROOT = Path(CONFIG["output_root"])
CACHE_ROOT = OUTPUT_ROOT / "cache"
STRUCTURED_ROOT = OUTPUT_ROOT / "structured_chapters"
PACK_ROOT = OUTPUT_ROOT / "generated_pack"
for path in [OUTPUT_ROOT, CACHE_ROOT, STRUCTURED_ROOT, PACK_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print("Environment ready")

## 02_PDF_DATASET_DISCOVERY

Scan the Kaggle dataset for raw PDF textbooks.

In [ ]:
PDF_RE = re.compile(r"\.pdf$", re.I)
CLASS_RE = re.compile(r"\bclass\s*([1-9]|10)\b", re.I)
SUBJECT_ALIASES = {
    "math": "mathematics",
    "maths": "mathematics",
    "mathematics": "mathematics",
    "science": "science",
    "social science": "social_science",
    "social_science": "social_science",
    "socialscience": "social_science",
    "biology": "biology",
    "physics": "physics",
    "chemistry": "chemistry",
    "history": "history",
    "geography": "geography",
    "economics": "economics",
    "civics": "civics",
}

def norm_subject(value: str) -> str:
    key = str(value or "").lower().replace("-", "_").replace(" ", "_").strip("_")
    return SUBJECT_ALIASES.get(key, key)

def candidate_roots(root: str) -> List[Path]:
    base = Path(root)
    roots = []
    if base.exists():
        roots.append(base)
        roots.extend(sorted([p for p in base.rglob("TEXTBOOKS") if p.is_dir()], key=lambda p: str(p).lower()))
    # Kaggle fallback: if configured root is wrong, scan /kaggle/input.
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists() and kaggle_input not in roots:
        roots.append(kaggle_input)
        roots.extend(sorted([p for p in kaggle_input.rglob("TEXTBOOKS") if p.is_dir()], key=lambda p: str(p).lower()))
    # Prefer more specific TEXTBOOKS roots first, then broader roots.
    return sorted(set(roots), key=lambda p: ("TEXTBOOKS" not in p.parts, len(p.parts)))

def infer_grade(path: Path) -> str:
    for part in path.parts:
        m = CLASS_RE.search(part)
        if m:
            return m.group(1)
    return ""

def infer_subject(path: Path) -> str:
    parts = [p.lower().replace("_", " ") for p in path.parts]
    for raw in ["mathematics", "maths", "math", "science", "social science", "social_science", "biology", "physics", "chemistry", "history", "geography", "economics", "civics"]:
        normalized = raw.replace("_", " ")
        if normalized in parts:
            return norm_subject(raw)
    # For layout TEXTBOOKS/SUBJECT/PDF or TEXTBOOKS/SUBJECT/class X/PDF.
    try:
        idx = [p.lower() for p in path.parts].index("textbooks")
        if idx + 1 < len(path.parts):
            return norm_subject(path.parts[idx + 1])
    except ValueError:
        pass
    return norm_subject(path.parent.name)

def passes_filters(path: Path) -> bool:
    include_subjects = {norm_subject(s) for s in CONFIG.get("include_subjects", []) if str(s).strip()}
    include_grades = {str(g) for g in CONFIG.get("include_grades", []) if str(g).strip()}
    name_filter = str(CONFIG.get("include_pdf_name_contains") or "").lower().strip()
    if include_subjects and infer_subject(path) not in include_subjects:
        return False
    if include_grades and infer_grade(path) not in include_grades:
        return False
    if name_filter and name_filter not in path.name.lower():
        return False
    return True

def discover_pdfs(root: str) -> List[Path]:
    all_files = []
    debug = []
    for base in candidate_roots(root):
        files = sorted([p for p in base.rglob("*") if p.is_file() and PDF_RE.search(p.name)], key=lambda p: str(p).lower())
        debug.append({"root": str(base), "pdf_count_before_filters": len(files)})
        all_files.extend(files)
    unique = sorted(set(all_files), key=lambda p: str(p).lower())
    filtered = [p for p in unique if passes_filters(p)]
    if CONFIG.get("max_pdfs"):
        filtered = filtered[: int(CONFIG["max_pdfs"])]
    if CONFIG.get("debug_discovery"):
        print("Discovery roots:")
        for row in debug[:20]:
            print(row)
        print("Sample discovered PDFs after filters:")
        for p in filtered[:10]:
            print(" -", p)
    return filtered

PDF_FILES = discover_pdfs(CONFIG["pdf_dataset_root"])
inventory = {
    "pdf_dataset_root": CONFIG["pdf_dataset_root"],
    "pdf_count": len(PDF_FILES),
    "grades": sorted(set(infer_grade(p) for p in PDF_FILES if infer_grade(p)), key=lambda x: int(x) if x.isdigit() else 999),
    "subjects": sorted(set(infer_subject(p) for p in PDF_FILES if infer_subject(p))),
    "sample_pdfs": [str(p) for p in PDF_FILES[:20]],
    "active_filters": {
        "include_subjects": CONFIG.get("include_subjects"),
        "include_grades": CONFIG.get("include_grades"),
        "include_pdf_name_contains": CONFIG.get("include_pdf_name_contains"),
        "max_pdfs": CONFIG.get("max_pdfs"),
    }
}
(OUTPUT_ROOT / "PDF_DATASET_INVENTORY.json").write_text(json.dumps(inventory, indent=2), encoding="utf-8")
inventory


## 03_DOCLING_PDF_EXTRACTION

Docling parses raw PDFs into structured chapter records. Outputs are cached as `structured_chapter.json` so interrupted Kaggle sessions can resume.

In [ ]:
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

WORD_RE = re.compile(r"[A-Za-z0-9]+(?:[-'][A-Za-z0-9]+)?")
FORMULA_RE = re.compile(r"([A-Za-z][A-Za-z0-9_ ()]{0,40}\s*(?:=|∝|≤|≥|<|>)\s*[^.;\n]{1,120}|[A-Z][a-z]?\s*=\s*[^.;\n]{1,120})")

def slugify(value: str) -> str:
    text = str(value or "").lower().strip()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return re.sub(r"_+", "_", text).strip("_") or "untitled"

def word_count(text: str) -> int:
    return len(WORD_RE.findall(text or ""))

def dump_docling_model(value: Any) -> Dict[str, Any]:
    if hasattr(value, "export_to_dict"):
        return value.export_to_dict()
    if hasattr(value, "model_dump"):
        return value.model_dump(mode="json")
    return value if isinstance(value, dict) else {}

def make_docling_converter() -> DocumentConverter:
    options = PdfPipelineOptions()
    options.do_ocr = bool(CONFIG["docling_do_ocr"])
    options.do_table_structure = bool(CONFIG["docling_do_table_structure"])
    options.generate_picture_images = bool(CONFIG["docling_generate_picture_images"])
    options.generate_page_images = bool(CONFIG["docling_generate_page_images"])
    options.do_picture_classification = False
    options.do_picture_description = False
    options.do_formula_enrichment = False
    return DocumentConverter(format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=options)})

DOCLING_CONVERTER = make_docling_converter()

def item_text(item: Dict[str, Any]) -> str:
    for key in ["text", "orig", "content"]:
        value = item.get(key)
        if isinstance(value, str) and value.strip():
            return re.sub(r"\s+", " ", value).strip()
    return ""

def table_rows(table: Dict[str, Any]) -> List[List[str]]:
    data = table.get("data") if isinstance(table.get("data"), dict) else table
    cells = data.get("table_cells") or data.get("cells") or []
    by_row: Dict[int, List[Tuple[int, str]]] = {}
    for cell in cells if isinstance(cells, list) else []:
        if not isinstance(cell, dict):
            continue
        row = int(cell.get("start_row_offset_idx") or cell.get("row") or 0)
        col = int(cell.get("start_col_offset_idx") or cell.get("col") or 0)
        text = str(cell.get("text") or "").strip()
        by_row.setdefault(row, []).append((col, text))
    return [[text for _, text in sorted(values)] for _, values in sorted(by_row.items())]

def pdf_output_path(pdf_path: Path) -> Path:
    grade = f"grade_{infer_grade(pdf_path) or 'unknown'}"
    subject = slugify(infer_subject(pdf_path) or "unknown")
    chapter = slugify(pdf_path.stem)
    return STRUCTURED_ROOT / grade / subject / chapter / "structured_chapter.json"

def convert_pdf_to_structured_chapter(pdf_path: Path) -> Dict[str, Any]:
    out_path = pdf_output_path(pdf_path)
    if CONFIG["resume"] and out_path.exists():
        return json.loads(out_path.read_text(encoding="utf-8"))

    result = DOCLING_CONVERTER.convert(str(pdf_path))
    doc = dump_docling_model(result.document)
    texts = doc.get("texts") if isinstance(doc.get("texts"), list) else []
    tables = doc.get("tables") if isinstance(doc.get("tables"), list) else []
    pictures = doc.get("pictures") if isinstance(doc.get("pictures"), list) else []

    blocks: List[Dict[str, Any]] = []
    for item in texts:
        if not isinstance(item, dict):
            continue
        text = item_text(item)
        if not text:
            continue
        label = str(item.get("label") or item.get("type") or "").lower()
        kind = "heading" if "title" in label or "section_header" in label else "paragraph"
        equations = [m.group(0).strip() for m in FORMULA_RE.finditer(text)]
        if equations:
            kind = "formula"
        blocks.append({"type": kind, "text": text, "equations": equations, "label": label})

    table_objs = []
    for idx, table in enumerate(tables, start=1):
        if isinstance(table, dict):
            rows = table_rows(table)
            if rows:
                table_objs.append({"title": str(table.get("caption") or f"Table {idx}"), "headers": rows[0], "rows": rows[1:]})

    figure_objs = []
    for idx, picture in enumerate(pictures, start=1):
        caption = ""
        if isinstance(picture, dict):
            caption = str(picture.get("caption") or picture.get("text") or "")
        figure_objs.append({"figure_id": f"figure_{idx}", "caption": caption})

    sections = build_sections_from_blocks(pdf_path.stem, blocks, table_objs, figure_objs)
    chapter = {
        "document_id": slugify(f"grade_{infer_grade(pdf_path)}_{infer_subject(pdf_path)}_{pdf_path.stem}"),
        "document_title": str(doc.get("name") or pdf_path.stem),
        "chapter_title": pdf_path.stem,
        "grade": infer_grade(pdf_path),
        "subject": infer_subject(pdf_path),
        "source_pdf": str(pdf_path),
        "sections": sections,
    }
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(chapter, indent=2, ensure_ascii=False), encoding="utf-8")
    return chapter

def build_sections_from_blocks(chapter_title: str, blocks: List[Dict[str, Any]], tables: List[Dict[str, Any]], figures: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    sections = []
    current_title = chapter_title
    current_parts: List[str] = []
    current_equations: List[str] = []

    def flush():
        if not current_parts:
            return
        content = "\n\n".join(current_parts).strip()
        section_id = hashlib.sha256((current_title + content).encode("utf-8")).hexdigest()[:16]
        sections.append({
            "section_id": f"section_{section_id}",
            "title": current_title,
            "level": 1,
            "content": content,
            "tables": [],
            "figures": [],
            "equations": sorted(set(current_equations)),
            "subsections": [],
            "metadata": {"word_count": word_count(content), "character_count": len(content), "estimated_tokens": max(1, len(content)//4)},
        })

    for block in blocks:
        text = block.get("text", "")
        if block.get("type") == "heading" and current_parts and word_count("\n".join(current_parts)) >= int(CONFIG["min_section_words"]):
            flush()
            current_parts = []
            current_equations = []
            current_title = text[:120]
        else:
            current_parts.append(text)
            current_equations.extend(block.get("equations", []))
            if word_count("\n".join(current_parts)) >= int(CONFIG["max_section_words"]):
                flush()
                current_parts = []
                current_equations = []
    flush()
    if sections:
        sections[0]["tables"] = tables
        sections[0]["figures"] = figures
    return sections

STRUCTURED_CHAPTERS = []
for pdf_path in tqdm(PDF_FILES, desc="Docling PDF extraction"):
    STRUCTURED_CHAPTERS.append(convert_pdf_to_structured_chapter(pdf_path))

(OUTPUT_ROOT / "DOCLING_EXTRACTION_MANIFEST.json").write_text(json.dumps({"chapters": len(STRUCTURED_CHAPTERS)}, indent=2), encoding="utf-8")
{"structured_chapters": len(STRUCTURED_CHAPTERS)}

## 04_MODEL_LOADING

Load Gemma 4 12B IT GGUF through llama.cpp. There is no separate vision model.

In [ ]:
from llama_cpp import Llama

def load_content_model(repo_id: str, filename: str) -> Llama:
    logger.info("Loading Gemma content model repo=%s file=%s", repo_id, filename)
    return Llama.from_pretrained(
        repo_id=repo_id,
        filename=filename,
        n_ctx=int(CONFIG["context_size"]),
        n_gpu_layers=int(CONFIG["n_gpu_layers"]),
        n_threads=int(CONFIG["n_threads"]),
        verbose=False,
    )

try:
    CONTENT_LLM = load_content_model(CONFIG["content_model_repo"], CONFIG["content_model_file"])
    ACTIVE_CONTENT_MODEL = {"repo": CONFIG["content_model_repo"], "file": CONFIG["content_model_file"], "fallback": False}
except Exception as primary_exc:
    logger.warning("Primary Gemma model failed: %s", primary_exc)
    CONTENT_LLM = load_content_model(CONFIG["fallback_content_model_repo"], CONFIG["fallback_content_model_file"])
    ACTIVE_CONTENT_MODEL = {"repo": CONFIG["fallback_content_model_repo"], "file": CONFIG["fallback_content_model_file"], "fallback": True, "primary_error": str(primary_exc)}
ACTIVE_CONTENT_MODEL

## 05_DATA_MODELS

In [ ]:
@dataclass
class FigureData:
    figure_id: str
    caption: str = ""

@dataclass
class SectionData:
    section_id: str
    title: str
    content: str
    tables: List[Dict[str, Any]] = field(default_factory=list)
    figures: List[FigureData] = field(default_factory=list)
    equations: List[str] = field(default_factory=list)
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class ChapterData:
    document_id: str
    document_title: str
    chapter_title: str
    grade: str
    subject: str
    sections: List[SectionData]
    source_pdf: str = ""

@dataclass
class ArtifactBundle:
    chapter_id: str
    artifacts: Dict[str, Any] = field(default_factory=dict)
    failures: List[Dict[str, Any]] = field(default_factory=list)

## 06_SECTION_ADAPTER

Convert Docling structured chapter dictionaries into reusable dataclasses.

In [ ]:
def adapt_section(raw: Dict[str, Any]) -> SectionData:
    return SectionData(
        section_id=str(raw.get("section_id") or hashlib.sha256(str(raw).encode()).hexdigest()[:16]),
        title=str(raw.get("title") or "Untitled Section"),
        content=str(raw.get("content") or ""),
        tables=list(raw.get("tables") or []),
        figures=[FigureData(figure_id=str(f.get("figure_id") or ""), caption=str(f.get("caption") or "")) for f in raw.get("figures", [])],
        equations=list(raw.get("equations") or []),
        metadata=dict(raw.get("metadata") or {}),
    )

def adapt_chapter(raw: Dict[str, Any]) -> ChapterData:
    sections = [adapt_section(s) for s in raw.get("sections", [])]
    if CONFIG["max_sections"]:
        sections = sections[: int(CONFIG["max_sections"])]
    return ChapterData(
        document_id=str(raw.get("document_id") or hashlib.sha256(str(raw).encode()).hexdigest()[:16]),
        document_title=str(raw.get("document_title") or "Untitled Document"),
        chapter_title=str(raw.get("chapter_title") or raw.get("document_title") or "Untitled Chapter"),
        grade=str(raw.get("grade") or ""),
        subject=str(raw.get("subject") or ""),
        sections=sections,
        source_pdf=str(raw.get("source_pdf") or ""),
    )

CHAPTERS = [adapt_chapter(chapter) for chapter in STRUCTURED_CHAPTERS]
load_report = {
    "chapters": len(CHAPTERS),
    "sections": sum(len(ch.sections) for ch in CHAPTERS),
    "empty_sections": [s.section_id for ch in CHAPTERS for s in ch.sections if not s.content.strip()],
}
(OUTPUT_ROOT / "LOAD_VALIDATION_REPORT.json").write_text(json.dumps(load_report, indent=2), encoding="utf-8")
load_report

## 07_SECTION_ENRICHMENT

Gemma receives Docling-extracted section text plus tables, figure captions, and equations. No separate vision model is used.

In [ ]:
def enriched_section_text(section: SectionData) -> str:
    parts = [f"Section Title: {section.title}", section.content]
    if section.equations:
        parts.append("Equations:\n" + "\n".join(section.equations[:20]))
    if section.tables:
        table_lines = []
        for table in section.tables[:5]:
            table_lines.append(json.dumps(table, ensure_ascii=False)[:2500])
        parts.append("Tables:\n" + "\n".join(table_lines))
    captions = [f"{fig.figure_id}: {fig.caption}" for fig in section.figures if fig.caption]
    if captions:
        parts.append("Figure references and captions:\n" + "\n".join(captions[:20]))
    return "\n\n".join(part for part in parts if part).strip()

## 08-21_ARTIFACT_GENERATION

Gemma section-level JSON generation with cache, retries, and validation.

In [ ]:
ARTIFACT_SPECS: Dict[str, Dict[str, Any]] = {
    "concepts": {"enabled": "generate_concepts", "file": "concepts.json", "schema": {"type": "object", "required": ["items"], "properties": {"items": {"type": "array", "items": {"type": "object", "required": ["concept", "evidence"], "properties": {"concept": {"type": "string"}, "evidence": {"type": "string"}, "difficulty": {"type": "string"}}}}}}},
    "learning_objectives": {"enabled": "generate_learning_objectives", "file": "learning_objectives.json", "schema": {"type": "object", "required": ["items"], "properties": {"items": {"type": "array", "items": {"type": "object", "required": ["objective"], "properties": {"objective": {"type": "string"}}}}}}},
    "glossary": {"enabled": "generate_glossary", "file": "glossary.json", "schema": {"type": "object", "required": ["items"], "properties": {"items": {"type": "array", "items": {"type": "object", "required": ["term", "definition"], "properties": {"term": {"type": "string"}, "definition": {"type": "string"}}}}}}},
    "summary": {"enabled": "generate_summary", "file": "summary.json", "schema": {"type": "object", "required": ["title", "summary", "key_points"], "properties": {"title": {"type": "string"}, "summary": {"type": "string"}, "key_points": {"type": "array", "items": {"type": "string"}}, "important_facts": {"type": "array", "items": {"type": "string"}}}}},
    "detailed_explanation": {"enabled": "generate_detailed_explanation", "file": "detailed_explanation.json", "schema": {"type": "object", "required": ["explanation"], "properties": {"explanation": {"type": "string"}, "teaching_steps": {"type": "array", "items": {"type": "string"}}}}},
    "misconceptions": {"enabled": "generate_misconceptions", "file": "misconceptions.json", "schema": {"type": "object", "required": ["items"], "properties": {"items": {"type": "array", "items": {"type": "object", "required": ["misconception", "correction"], "properties": {"misconception": {"type": "string"}, "correction": {"type": "string"}}}}}}},
    "mcq_quiz": {"enabled": "generate_mcq_quiz", "file": "mcq_quiz.json", "schema": {"type": "object", "required": ["items"], "properties": {"items": {"type": "array", "items": {"type": "object", "required": ["question", "options", "answer", "explanation", "difficulty"], "properties": {"question": {"type": "string"}, "options": {"type": "array", "minItems": 4, "maxItems": 4, "items": {"type": "string"}}, "answer": {"type": "string"}, "explanation": {"type": "string"}, "difficulty": {"type": "string"}}}}}}},
    "short_answer_questions": {"enabled": "generate_short_answer", "file": "short_answer_questions.json", "schema": {"type": "object", "required": ["items"], "properties": {"items": {"type": "array", "items": {"type": "object", "required": ["question", "answer", "rubric"], "properties": {"question": {"type": "string"}, "answer": {"type": "string"}, "rubric": {"type": "string"}}}}}}},
    "flashcards": {"enabled": "generate_flashcards", "file": "flashcards.json", "schema": {"type": "object", "required": ["items"], "properties": {"items": {"type": "array", "items": {"type": "object", "required": ["question", "answer", "difficulty"], "properties": {"question": {"type": "string"}, "answer": {"type": "string"}, "difficulty": {"type": "string"}}}}}}},
    "concept_relationships": {"enabled": "generate_concept_relationships", "file": "concept_relationships.json", "schema": {"type": "object", "required": ["items"], "properties": {"items": {"type": "array", "items": {"type": "object", "required": ["source", "target", "relationship"], "properties": {"source": {"type": "string"}, "target": {"type": "string"}, "relationship": {"type": "string"}}}}}}},
    "investigations": {"enabled": "generate_investigations", "file": "investigations.json", "schema": {"type": "object", "required": ["items"], "properties": {"items": {"type": "array", "items": {"type": "object", "required": ["title", "hypothesis", "variables", "observations"], "properties": {"title": {"type": "string"}, "hypothesis": {"type": "string"}, "variables": {"type": "array", "items": {"type": "string"}}, "observations": {"type": "array", "items": {"type": "string"}}}}}}}},
    "teacher_notes": {"enabled": "generate_teacher_notes", "file": "teacher_notes.json", "schema": {"type": "object", "required": ["notes"], "properties": {"notes": {"type": "array", "items": {"type": "string"}}, "classroom_tips": {"type": "array", "items": {"type": "string"}}}}},
    "prerequisites": {"enabled": "generate_prerequisites", "file": "prerequisites.json", "schema": {"type": "object", "required": ["items"], "properties": {"items": {"type": "array", "items": {"type": "object", "required": ["concept", "prerequisite"], "properties": {"concept": {"type": "string"}, "prerequisite": {"type": "string"}}}}}}},
    "difficulty_analysis": {"enabled": "generate_difficulty_analysis", "file": "difficulty_analysis.json", "schema": {"type": "object", "required": ["difficulty", "reasons"], "properties": {"difficulty": {"type": "string"}, "reasons": {"type": "array", "items": {"type": "string"}}, "estimated_grade_fit": {"type": "string"}}}},
}

def extract_json_object(text: str) -> Any:
    text = re.sub(r"^```(?:json)?", "", text.strip(), flags=re.I).strip()
    text = re.sub(r"```$", "", text).strip()
    decoder = json.JSONDecoder()
    for start, char in enumerate(text):
        if char in "[{":
            try:
                parsed, _ = decoder.raw_decode(text[start:])
                return parsed
            except json.JSONDecodeError:
                continue
    raise ValueError("No JSON object found")

def prompt_for_artifact(section: SectionData, artifact_name: str, schema: Dict[str, Any]) -> str:
    return f"""
You are generating curriculum artifacts for offline school tutoring.
Use only the Docling-extracted source section, including tables, equations, and figure captions.
Do not invent facts. Return valid JSON only. No markdown.
Artifact: {artifact_name}
Required JSON Schema: {json.dumps(schema, ensure_ascii=False)}
Source:
{enriched_section_text(section)[:12000]}
""".strip()

def cache_path(section: SectionData, artifact_name: str) -> Path:
    digest = hashlib.sha256((section.section_id + artifact_name + section.content).encode("utf-8")).hexdigest()
    return CACHE_ROOT / f"{digest}.json"

def generate_json(section: SectionData, artifact_name: str, schema: Dict[str, Any]) -> Dict[str, Any]:
    cpath = cache_path(section, artifact_name)
    if CONFIG["resume"] and cpath.exists():
        return json.loads(cpath.read_text(encoding="utf-8"))
    last_error = ""
    base_prompt = prompt_for_artifact(section, artifact_name, schema)
    for retry in range(int(CONFIG["max_retries"])):
        prompt = base_prompt if retry == 0 else base_prompt + f"\nPrevious JSON was invalid: {last_error}. Return corrected JSON only."
        result = CONTENT_LLM(prompt, max_tokens=int(CONFIG["max_tokens"]), temperature=float(CONFIG["temperature"]), top_p=float(CONFIG["top_p"]), stop=["```"])
        raw = result["choices"][0]["text"]
        try:
            parsed = extract_json_object(raw)
            if CONFIG["enable_validation"]:
                jsonschema_validate(parsed, schema)
            cpath.write_text(json.dumps(parsed, indent=2, ensure_ascii=False), encoding="utf-8")
            return parsed
        except Exception as exc:
            last_error = str(exc)
    raise RuntimeError(f"Generation failed for {artifact_name}/{section.section_id}: {last_error}")

## 22_VALIDATION_QUALITY_EXPORT

In [ ]:
def append_artifact(target: Dict[str, List[Any]], artifact_name: str, section: SectionData, payload: Dict[str, Any]) -> None:
    target.setdefault(artifact_name, []).append({"section_id": section.section_id, "section_title": section.title, "payload": payload})

def validate_pack_artifacts(artifacts: Dict[str, List[Any]]) -> Dict[str, Any]:
    report = {"artifact_counts": {}, "empty_artifacts": [], "duplicate_questions": 0, "missing_answers": 0}
    seen_questions = set()
    for artifact_name, entries in artifacts.items():
        report["artifact_counts"][artifact_name] = len(entries)
        if not entries:
            report["empty_artifacts"].append(artifact_name)
        for entry in entries:
            payload = entry.get("payload", {})
            for item in payload.get("items", []) if isinstance(payload, dict) else []:
                question = str(item.get("question", "")).strip().lower()
                if question:
                    if question in seen_questions:
                        report["duplicate_questions"] += 1
                    seen_questions.add(question)
                if "question" in item and not item.get("answer"):
                    report["missing_answers"] += 1
    return report

def quality_metrics(artifacts: Dict[str, List[Any]], failures: List[Dict[str, Any]]) -> Dict[str, Any]:
    enabled = [name for name, spec in ARTIFACT_SPECS.items() if CONFIG.get(spec["enabled"], False)]
    covered = [name for name in enabled if artifacts.get(name)]
    return {"enabled_artifacts": enabled, "artifact_coverage": round(len(covered) / max(1, len(enabled)), 4), "generation_failures": len(failures), "model": ACTIVE_CONTENT_MODEL}

run_started = time.time()
all_artifacts: Dict[str, List[Any]] = {}
failures: List[Dict[str, Any]] = []
sections_processed = 0

for chapter in tqdm(CHAPTERS, desc="Chapters"):
    for section in tqdm(chapter.sections, desc=f"Sections: {chapter.chapter_title}", leave=False):
        sections_processed += 1
        for artifact_name, spec in ARTIFACT_SPECS.items():
            if not CONFIG.get(spec["enabled"], False):
                continue
            try:
                payload = generate_json(section, artifact_name, spec["schema"])
                append_artifact(all_artifacts, artifact_name, section, payload)
            except Exception as exc:
                failures.append({"chapter": chapter.chapter_title, "section_id": section.section_id, "artifact": artifact_name, "error": str(exc)})
                logger.exception("Generation failed: %s %s", section.section_id, artifact_name)

for artifact_name, spec in ARTIFACT_SPECS.items():
    if CONFIG.get(spec["enabled"], False):
        (PACK_ROOT / spec["file"]).write_text(json.dumps(all_artifacts.get(artifact_name, []), indent=2, ensure_ascii=False), encoding="utf-8")

validation_report = validate_pack_artifacts(all_artifacts)
quality_report = quality_metrics(all_artifacts, failures)
run_report = {"chapters_processed": len(CHAPTERS), "sections_processed": sections_processed, "generation_failures": len(failures), "elapsed_seconds": round(time.time() - run_started, 2), "failures": failures[:200]}

(PACK_ROOT / "VALIDATION_REPORT.json").write_text(json.dumps(validation_report, indent=2, ensure_ascii=False), encoding="utf-8")
(PACK_ROOT / "QUALITY_REPORT.json").write_text(json.dumps(quality_report, indent=2, ensure_ascii=False), encoding="utf-8")
(PACK_ROOT / "RUN_REPORT.json").write_text(json.dumps(run_report, indent=2, ensure_ascii=False), encoding="utf-8")

zip_path = OUTPUT_ROOT / CONFIG["artifact_zip_name"]
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for file_path in PACK_ROOT.rglob("*.json"):
        archive.write(file_path, arcname=str(file_path.relative_to(PACK_ROOT.parent)))

{"zip": str(zip_path), "run_report": run_report, "quality_report": quality_report, "validation_report": validation_report}

## Detailed Implementation Notes

### What this notebook implements

1. **Raw PDF input**
   - The dataset root points to Kaggle input containing textbook PDFs.
   - The notebook discovers all `*.pdf` files recursively.
   - It infers grade and subject from folder names where possible.

2. **Docling extraction inside Kaggle**
   - The notebook installs and uses `docling` directly.
   - It converts raw PDFs into structured chapter dictionaries.
   - It preserves text blocks, tables, figure references/captions, and equation-like strings.
   - It caches each extracted chapter under `structured_chapters/` so repeated runs resume.

3. **No separate vision model**
   - There is no Qwen or other vision model dependency.
   - Gemma receives Docling-extracted text, tables, captions, and equations.
   - If Docling extracts figure captions, those captions are included in Gemma prompts.
   - Raw image understanding is intentionally out of scope for this notebook.

4. **Gemma 4 12B content generation**
   - The content model is configured as `lmstudio-community/gemma-4-12B-it-GGUF`.
   - The file is configured as `gemma-4-12B-it-Q4_K_M.gguf`.
   - Loading is done through `llama-cpp-python` with `Llama.from_pretrained`.
   - A smaller Gemma fallback is configurable but only used if primary loading fails.

5. **Section-level processing**
   - The notebook never sends an entire book to Gemma.
   - Docling blocks are grouped into bounded sections.
   - Each section is independently usable as a prompt.

6. **Generated artifacts**
   - `concepts.json`
   - `learning_objectives.json`
   - `glossary.json`
   - `summary.json`
   - `detailed_explanation.json`
   - `misconceptions.json`
   - `mcq_quiz.json`
   - `short_answer_questions.json`
   - `flashcards.json`
   - `concept_relationships.json`
   - `investigations.json`
   - `teacher_notes.json`
   - `prerequisites.json`
   - `difficulty_analysis.json`

7. **Validation and recovery**
   - Each artifact has a JSON schema.
   - Invalid JSON is retried with repair instructions.
   - Per-section artifact outputs are cached.
   - The notebook supports resume after Kaggle timeouts or failures.

8. **Export**
   - All artifact JSON files are written to `generated_pack/`.
   - `VALIDATION_REPORT.json`, `QUALITY_REPORT.json`, and `RUN_REPORT.json` are generated.
   - The final output is zipped as `generated_pack.zip`.

### Required Kaggle dataset structure

Recommended:

```text
/kaggle/input/idp-textbooks/
  TEXTBOOKS/
    mathematics/
      class 8/
        Chapter.pdf
    science/
      class 10/
        Chapter.pdf
```

Any nested structure works as long as raw PDFs are present.

### Runtime guidance

- Gemma 4 12B Q4_K_M is large. Use Kaggle GPU. Select **T4** in the Kaggle UI.
- T4 is recommended over P100 here for newer CUDA/runtime compatibility with modern llama.cpp stacks. P100 has high memory bandwidth, but T4 is usually the safer Kaggle target for this workflow.
- Start with `max_pdfs=1` or `max_sections=2` for smoke testing.
- Then increase gradually.
- Keep `resume=True` so completed generations are not repeated.